# 🔬 TensorRT-LLM 源码级深度剖析

> **核心命题**：如何通过编译优化，将 LLM 推理加速到接近硬件极限？

TensorRT-LLM 代表了推理优化的另一个极端——不是改进内存管理（vLLM），而是**编译时做尽可能多的优化**，
让运行时尽可能少做决策。

## 哲学对比

```
llama.cpp:  运行时灵活（任何模型、任何量化）→ 每次推理都有解释开销
vLLM:       运行时调度（动态 batch、动态分配）→ 调度本身有开销
TensorRT-LLM: 编译时决定一切 → 运行时零开销，但模型必须预编译

类比：
  llama.cpp = Python 解释器（灵活，每次执行时有 JIT 开销）
  TensorRT-LLM = AOT 编译的 C 程序（不灵活，但执行最快）
  vLLM = JIT 编译（在两者之间）
```

## 1. 架构全景

```
┌──────────────────────────────────────────────────────────────┐
│                    TensorRT-LLM 架构                         │
├──────────────────────────────────────────────────────────────┤
│                                                               │
│  ┌─────────────────────────────────────────────────────────┐ │
│  │              Model Definition API (Python)               │ │
│  │  • 用户用 Python 描述模型（类似 PyTorch 但用于编译）      │ │
│  │  • tensorrt_llm.Builder / tensorrt_llm.models           │ │
│  └─────────────────────────┬───────────────────────────────┘ │
│                            │                                  │
│  ┌─────────────────────────▼───────────────────────────────┐ │
│  │              Graph Optimizer (图优化)                    │ │
│  │  • Layer Fusion (算子融合): LayerNorm + Attention       │ │
│  │  • Kernel Auto-tuning: 自动选择最佳 GEMM 配置           │ │
│  │  • Quantization: FP8/INT8/INT4 直接编译到引擎           │ │
│  │  • Memory Planning: 预分配所有显存                      │ │
│  └─────────────────────────┬───────────────────────────────┘ │
│                            │                                  │
│  ┌─────────────────────────▼───────────────────────────────┐ │
│  │              TensorRT Engine (序列化)                    │ │
│  │  • 编译产物：包含所有 kernel、权重、内存布局的二进制     │ │
│  │  • .engine 文件：可跨相同 GPU 架构复用                   │ │
│  └─────────────────────────┬───────────────────────────────┘ │
│                            │                                  │
│  ┌─────────────────────────▼───────────────────────────────┐ │
│  │              Runtime (C++)                               │ │
│  │  • In-flight Batching Scheduler                         │ │
│  │  • KV Cache Manager (类似 PagedAttention)               │ │
│  │  • 多 GPU Tensor Parallelism + Pipeline Parallelism     │ │
│  └─────────────────────────────────────────────────────────┘ │
│                                                               │
└──────────────────────────────────────────────────────────────┘
```

## 2. 编译流程：从 HuggingFace 到 .engine

```python
# 完整的 TensorRT-LLM 模型构建流程

# Step 1: 定义模型架构
from tensorrt_llm import Builder
from tensorrt_llm.network import net_guard
from tensorrt_llm.functional import *

builder = Builder()

with net_guard(builder.create_network()):
    # 定义输入
    input_ids = INPUT('input_ids', dtype=DataType.INT32, shape=(-1, -1))

    # Embedding
    hidden_states = embedding(input_ids, vocab_size, hidden_size, dtype)

    # Transformer layers — 每层都会被图优化器分析和融合
    for layer_idx in range(num_layers):
        # Attention block
        residual = hidden_states
        hidden_states = layer_norm(hidden_states)

        Q = matmul(hidden_states, Wq[layer_idx])
        K = matmul(hidden_states, Wk[layer_idx])
        V = matmul(hidden_states, Wv[layer_idx])

        # 这里 TensorRT 会自动:
        # 1. 识别 Q/K/V 是三个相邻的 matmul → 融合成一个 batch matmul
        # 2. 如果用了 FP8 → 插入量化/反量化节点
        # 3. 如果用了 FlashAttention → 替换为 fused attention kernel

        context = attention(Q, K, V, mask, num_heads, head_size)
        hidden_states = matmul(context, Wo[layer_idx])
        hidden_states = hidden_states + residual

        # FFN block — SwiGLU 被识别并融合
        residual = hidden_states
        hidden_states = layer_norm(hidden_states)
        hidden_states = swiglu(hidden_states, W1[layer_idx], W2[layer_idx], W3[layer_idx])
        hidden_states = hidden_states + residual

    # LM head
    logits = matmul(hidden_states, lm_head)
    output = identity(logits)  # 标记输出

# Step 2: 编译
builder_config = BuilderConfig(
    max_batch_size=8,
    max_input_len=4096,
    max_output_len=2048,
    # FP8 量化 — 需要 H100/Ada 架构
    quantization_mode=QuantMode.FP8_KV_CACHE | QuantMode.FP8_QDQ,
    # In-flight batching
    plugin_config=PluginConfig(
        gpt_attention_plugin='float16',  # 使用优化的 attention kernel
        gemm_plugin='float16',           # 使用优化的 GEMM
    )
)

# Step 3: 序列化
engine = builder.build_engine(network, builder_config)
with open('llama3-8b-fp8.engine', 'wb') as f:
    f.write(engine)
```

## 3. 核心优化：算子融合

TensorRT 的 Graph Optimizer 做的最重要的事情就是算子融合。

```
融合前 (原始计算图，6 个 kernel):
  Q = MatMul(X, Wq)        ← Kernel 1: 读 X, Wq → 写 Q
  K = MatMul(X, Wk)        ← Kernel 2: 读 X, Wk → 写 K
  V = MatMul(X, Wv)        ← Kernel 3: 读 X, Wv → 写 V
  context = Attention(Q,K,V) ← Kernel 4: 读 Q,K,V → 写 context
  output = MatMul(context, Wo) ← Kernel 5: 读 context, Wo → 写 output

  问题：Q, K, V 是中间结果，写到显存再读回来 → 浪费带宽

融合后 (1 个 kernel):
  FusedAttention(X, Wq, Wk, Wv, Wo) → output
  ← 所有操作在一个 kernel 中完成
  ← Q, K, V 留在寄存器/shared memory，不写到显存

  带宽节省：Q, K, V 各 (batch × seq_len × hidden) × 2 (写+读)
```

**关键融合模式**：

| 融合名称 | 合并的算子 | 效果 |
|---------|----------|------|
| QKV Fusion | Wq + Wk + Wv → 1 个 GEMM | 权重加载减 3x |
| Attention Fusion | Attention + Output Projection | 消除 context 中间结果 |
| FFN Fusion | Gate + Up projection | GEMM 数量从 3 减到 2 |
| MLP Fusion | SwiGLU 全部 | 消除中间激活 |
| Residual Fusion | Add + LayerNorm | 消除一次显存往返 |

## 4. In-flight Batching：比 Continuous Batching 更进一步

```python
# TensorRT-LLM 的 In-flight Batching 调度器
# 对应的核心文件: cpp/tensorrt_llm/batch_manager/

class TrtGptModel:
    def forward(self, batch: Batch) -> Logits:
        """
        In-flight Batching 的关键：
        同一个 batch 中可以有「正在 prefill 的请求」和「正在 decode 的请求」混在一起。

        这是 vLLM 的 Continuous Batching 做不到的——
        vLLM 要求一个 batch 要么全是 prefill，要么全是 decode。
        """
        # batch 中的每个请求有自己的阶段:
        #   - context_phase: 需要 prefill 整个 prompt
        #   - generation_phase: 只需要 decode 一个 token
        mixed_batch = self.scheduler.prepare_batch()

        # 单个 forward 调用，内部处理不同的阶段
        return self.model(mixed_batch)

class BatchManager:
    def prepare_next_batch(self) -> Batch:
        """
        从队列中挑选请求，构建混合 batch。

        策略：
        1. 优先保证正在 decode 的请求能继续（SLA 保护，低延迟）
        2. 剩余的 token budget 分配给新的 prefill 请求（高吞吐）
        3. 如果 token budget 满了，prefill 请求排队
        """
        batch = Batch()
        token_budget = self.max_batch_tokens

        # 第一优先：正在 decode 的请求
        for req in self.running_decodes:
            batch.add(req, phase='decode')
            token_budget -= 1  # decode 每个请求只处理 1 token

        # 第二优先：抢占恢复的请求（SWAPPED 状态的）
        for req in self.swapped:
            if token_budget >= req.swap_blocks * BLOCK_SIZE:
                batch.add(req, phase='recompute')
                token_budget -= req.swap_blocks * BLOCK_SIZE

        # 剩余预算：新请求的 prefill
        for req in self.waiting:
            if token_budget >= len(req.prompt_tokens):
                batch.add(req, phase='context')
                token_budget -= len(req.prompt_tokens)

        return batch
```

**In-flight Batching 的核心优势**：

```
Continuous Batching (vLLM):
  Step 0: [Decode batch: R1, R2, R3]          ← 全是 decode
  Step 1: [Prefill batch: R4████████████]     ← 必须等 R4 prefill 完成
          R1, R2, R3 在这个 step 被阻塞！       ← 短请求被长 prefill 阻塞

In-flight Batching (TensorRT-LLM):
  Step 0: [R1(decode), R2(decode), R4██(prefill)]  ← 混合！
  Step 1: [R1(decode), R3(decode), R4██(prefill)]  ← R4 prefill 继续
          R2 完成了，立即释放。                      ← 短请求不受影响
  Step 2: [R1(decode), R5(decode), R4(decode)]    ← R4 prefill 完成，转为 decode
```

## 5. FP8 量化：硬件原生支持的威力

NVIDIA H100 引入了 FP8（float8）的硬件支持，TensorRT-LLM 是第一等公民。

```
FP16 × FP16 → FP32 accumulate (传统)
  vs
FP8 × FP8 → FP16 accumulate (H100 的 Transformer Engine)

H100 FP8 GEMM 吞吐:
  - FP16: 989 TFLOPS
  - FP8:  1979 TFLOPS  ← 翻倍！且是硬件原生支持，无精度损失

TensorRT-LLM 的 FP8 方案:
  1. 权重离线量化: FP16 → FP8 (per-tensor/per-channel scale)
  2. 激活在线量化: 每步动态计算 scale（FP8 的动态范围小，必须动态 scale）
  3. 反向传播不需要，推理只需要前向
```

## 6. TensorRT-LLM vs vLLM：什么时候选哪个？

| 维度 | TensorRT-LLM | vLLM |
|------|-------------|------|
| **编译时间** | 10-30 分钟（一次性） | 0（JIT） |
| **推理吞吐** | 最高（+20-50% vs vLLM） | 高 |
| **模型支持** | ~20 种主流模型 | 100+ 种 HuggingFace 模型 |
| **新模型适配** | 需要写 C++ plugin | Python 写几行配置 |
| **FP8 支持** | 原生（H100 硬件加速） | 通过 vLLM-FP8 插件 |
| **多 GPU** | 最佳（Tensor/Pipeline Parallelism） | 支持（Ray + TP） |
| **Latency** | 最低（编译时预分配所有资源） | 略有调度开销 |
| **部署复杂度** | 高（需要编译 + 特定 GPU 架构） | 低（pip install） |
| **灵活切换模型** | 需要重新编译 | 即时切换 |

**选型原则**：
- 模型固定、需要极致吞吐 → TensorRT-LLM
- 需要频繁切换模型/实验 → vLLM
- H100/H200 硬件 → 优先考虑 TRT-LLM（FP8 优势明显）
- 非 NVIDIA 硬件 → vLLM（TRT-LLM 绑定 CUDA）

## 下一步

- → `05-tgi-architecture.ipynb`：HuggingFace 的官方推理方案
- → 返回 `../00-overview.ipynb` 查看全景对比
- → `../distributed/06-sglang-deep-dive.ipynb`：另一个极端——极致的前缀复用